### Copyright (C) Infineon Technologies AG 2025
 
Copyright (c) 2025, Infineon Technologies AG, or an affiliate of Infineon Technologies AG. All rights reserved.
This software, associated documentation and materials ("Software") is owned by Infineon Technologies AG or one of its affiliates ("Infineon") and is protected by and subject to worldwide patent protection, worldwide copyright laws, and international treaty provisions. Therefore, you may use this Software only as provided in the license agreement accompanying the software package from which you obtained this Software. If no license agreement applies, then any use, reproduction, modification, translation, or compilation of this Software is prohibited without the express written permission of Infineon.

Disclaimer: UNLESS OTHERWISE EXPRESSLY AGREED WITH INFINEON, THIS SOFTWARE IS PROVIDED AS-IS, WITH NO WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING, BUT NOT LIMITED TO, ALL WARRANTIES OF NON-INFRINGEMENT OF THIRD-PARTY RIGHTS AND IMPLIED WARRANTIES SUCH AS WARRANTIES OF FITNESS FOR A SPECIFIC USE/PURPOSE OR MERCHANTABILITY. Infineon reserves the right to make changes to the Software without notice. You are responsible for properly designing, programming, and testing the functionality and safety of your intended application of the Software, as well as complying with any legal requirements related to its use. Infineon does not guarantee that the Software will be free from intrusion, data theft or loss, or other breaches ("Security Breaches"), and Infineon shall have no liability arising out of any Security Breaches. Unless otherwise explicitly approved by Infineon, the Software may not be used in any application where a failure of the Product or any consequences of the use thereof can reasonably be expected to result in personal injury.

### Notebook Structure

1. **Generation of the Model: CNN for Rain Drop Classification**
2. **Post Training Quantization**
3. **Compiling the Model for AURIX&trade; Microcontrollers**

# CNN for Weather Classification

This is an example of classification of different weather conditions. The model will be quantized, tested and then transformed into code which can later be deployed on the AURIX&trade; microcontroller family. 

### Generation of the Model

In this section, we will generate a CNN for the classification task.

In [ ]:
import os
import sys
from torchsummary import summary
import torch.optim as optim
import numpy as np

import modelling_helper as mh

parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from _CentralScripts.python_flask_client import CallTools
import _CentralScripts.helper_functions as cs
import _CentralScripts.mobilenet_helper as mo
import MobileNetV3ClassificationRainDrops.modelling_helper as mm

device = cs.get_device()

Fetching data aka setting dataloader. The data which is used is Raindrop Clarify [1]. While this was designed for removing rain drops from images algorithmically, it will here be used for classification only. The data will be downloaded and restructured. From the original dataset only, clear and (rain) drop will be considered. The data will be split into train, validation (val), and test piles. Plotting a random input.

In [ ]:
batch_size = 16
resolution = 150

path = mm.fetch_sort_data()
dataloader, class_names, dataset_sizes = mo.get_dataloader(
    os.path.join(path, "train"), batch_size, mode="train", resolution=resolution
)
_, input_size = mm.get_input_dataloader(dataloader, is_plot=True)

Generating a CNN for the classification of two classes (rain drop vs clear weather).

In [ ]:
import torch.nn as nn

model = mh.get_model(2)
summary(model, input_size, device="cpu")

Training the model.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
mh.train_model(model, dataloader, dataset_sizes, criterion, optimizer, num_epochs=5)

Getting a random input which can be used for testing on the hardware later.

In [ ]:
input_target, _ = mm.get_input_dataloader(dataloader, is_plot=True)
output_target = cs.get_predictions("torch", model, input_target)

Saving the model and some input and output data.

In [ ]:
model_name = "cnn_weather"
cs.save_all(model_name, input_target, output_target, model, origin="torch")

### Post-Training Quantization

Preprocessing the model to prepare it for quantization.

In [ ]:
model_folder, onnx_model_file = cs.get_output_paths(model_name)
model_path_preprocessed, model_path_quantized = mh.create_folders_quantization(
    model_name
)
mh.onnx_preprocessing(onnx_model_file, model_path_preprocessed)

Preparing calibration data for quantization.

In [ ]:
calib_samples, _ = mh.get_calib_data(path, resolution=resolution, num_samples=100)
input_name = mh.get_input_name(model_path_preprocessed)
calibration_data = mh.NumpyDataReader(input_name, calib_samples)

Quantizing the model.

In [ ]:
from onnxruntime.quantization import quantize_static, QuantType

quantize_static(
    model_input=model_path_preprocessed,
    model_output=model_path_quantized,
    calibration_data_reader=calibration_data,
    per_channel=False,  # per-channel shall be False
    activation_type=QuantType.QInt8,  # typical for activations
    weight_type=QuantType.QInt8,  # typical for weights
)

print(f"Saved QDQ model to: {model_path_quantized}")

Validating the quantized model by comparing its output with the ones of the original model.

In [ ]:
dataloader_val, class_names, dataset_sizes = mo.get_dataloader(
    os.path.join(path, "val"), batch_size=1, mode="val", resolution=resolution
)

mh.validate_quantization(model_path_quantized, onnx_model_file, dataloader_val)

Saving input and output samples for the quantized model.

In [ ]:
quantized_model_onnx = cs.get_onnx_tensor(model_path_quantized)
_, output_target = mh.predict_class(
    quantized_model_onnx, np.expand_dims(input_target, axis=0)
)

cs.save_data(
    model_path_quantized.replace("model.onnx", ""), input_target, is_input=True
)
cs.save_data(
    model_path_quantized.replace("model.onnx", ""), output_target, is_input=False
)

### Compiling the Model for AURIX&trade; Microcontrollers

Testing if the Docker container is available.

In [ ]:
cs.ensure_docker_container()

Converting the model.

In [ ]:
model_name_quantized = f"{model_name}_quantized"
model_folder, onnx_model_file = cs.get_output_paths(model_name_quantized)

print(model_folder)
for target in ["TC3", "TC4"]:
    tool = CallTools(
        folder=model_folder, url="http://localhost:8080/convert", target=target
    )
    tool.convert_model()

Plotting instruction counts of the model.

In [ ]:
cs.plot_instruction_counts(model_name_quantized)

[1]: Jin, Y. et al. (2024). "Raindrop Clarity: A Dual-Focused Dataset for Day and Night Raindrop Removal", ECCV, [Link to Github](https://github.com/jinyeying/RaindropClarity?tab=readme-ov-file)<br>